In [1]:
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
data_path = os.getenv("DATA_PATH")

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
df = pd.read_csv(data_path)

df.replace("?", np.nan, inplace=True)

for col in ['workclass', 'occupation', 'native.country']:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.drop_duplicates(inplace=True)

In [5]:
df_encoded = df.copy()

# Encode all categorical columns
for col in df_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

In [6]:
X = df_encoded.drop("income", axis=1)
y = df_encoded["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (26029, 14)
Testing size: (6508, 14)


We check whether:

The proportion of income >50K in training set
is statistically similar to
The proportion of income >50K in testing set

If similar → split is valid.

In [8]:
# Count of class 1 (>50K)
count_train = np.sum(y_train)
count_test = np.sum(y_test)

# Total samples
n_train = len(y_train)
n_test = len(y_test)

count = np.array([count_train, count_test])
nobs = np.array([n_train, n_test])

z_stat, p_value = proportions_ztest(count, nobs)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -0.0017968488567871026
P-value: 0.9985663228105508


Interpretation :- 

If p-value > 0.05
→ No significant difference
→ Train-test split is valid

If p-value < 0.05
→ Significant difference
→ Bad split

